# Silver Layer — Market Observations

## Purpose
Transforms Bronze observations into validated, typed, de-duplicated records.
Invalid rows go to `silver_quarantine` with a rejection reason; the batch never crashes on bad data.

## Parameters
| Widget | Values | Behaviour |
|---|---|---|
| `load_type` | `full` / `incremental` | `full` = all Bronze rows (optionally limited by `processing_date` / `source_folder`). `incremental` = Bronze rows for `processing_date` plus the previous day (the downloader's 1-day overlap). Blank date = today (UTC). |
| `processing_date` | `YYYY-MM-DD` or blank | Any historical date works, which gives a Bronze-to-Silver backfill. |
| `source_folder` | text or blank | Only Bronze rows whose `source_file` contains this text. |

## Target table `workspace.default.silver_market_observation`
Primary key: `coin_id` + `observation_timestamp`

| Column | Type | Description |
|---|---|---|
| coin_id | STRING (not null) | CoinGecko coin slug, PK part 1 |
| observation_timestamp | TIMESTAMP (not null) | UTC observation time, PK part 2 |
| price_usd | DOUBLE | Price in USD |
| market_cap_usd | DOUBLE | Market cap in USD |
| total_volume_usd | DOUBLE | Trading volume in USD |
| source_file | STRING | Raw file the row came from |
| bronze_load_timestamp | TIMESTAMP | When the row was ingested into Bronze |
| load_timestamp | TIMESTAMP (not null) | When Silver processed the row |
| run_id | STRING | Silver run that wrote the row |

## Data quality rules
Missing/blank coin id, missing timestamp, and null/NaN/negative price, market cap or volume are rejected.
Values are converted with `try_cast`, so a type change at the source (e.g. a number arriving as text) becomes a quarantined row, not a crash.

## Idempotency
1. Source is de-duplicated on the business key (latest Bronze row wins).
2. `MERGE` on the business key; matched rows are updated **only if a measure changed**.
3. Re-running the same input therefore gives 0 inserted and 0 updated rows.

In [ ]:
# Step 112: Silver setup and parameters
import json, os, sys, uuid
from datetime import date, datetime, timedelta, timezone

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType, LongType
)

# Explicit schemas (the contract for each table)
silver_schema = StructType([
    StructField("coin_id", StringType(), False),
    StructField("observation_timestamp", TimestampType(), False),
    StructField("price_usd", DoubleType(), True),
    StructField("market_cap_usd", DoubleType(), True),
    StructField("total_volume_usd", DoubleType(), True),
    StructField("source_file", StringType(), True),
    StructField("bronze_load_timestamp", TimestampType(), True),
    StructField("load_timestamp", TimestampType(), False),
    StructField("run_id", StringType(), True),
])

quarantine_schema = StructType([
    StructField("coin_id", StringType(), True),
    StructField("source_file", StringType(), True),
    StructField("timestamp_ms", LongType(), True),
    StructField("observation_timestamp", TimestampType(), True),
    StructField("price_usd", DoubleType(), True),
    StructField("market_cap_usd", DoubleType(), True),
    StructField("total_volume_usd", DoubleType(), True),
    StructField("rejection_reason", StringType(), False),
    StructField("bronze_load_timestamp", TimestampType(), True),
    StructField("load_timestamp", TimestampType(), False),
    StructField("run_id", StringType(), True),
])

execution_log_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("load_type", StringType(), True),
    StructField("processing_date", StringType(), True),
    StructField("source_folder", StringType(), True),
    StructField("start_timestamp", TimestampType(), True),
    StructField("end_timestamp", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("rows_read", LongType(), True),
    StructField("rows_inserted", LongType(), True),
    StructField("rows_updated", LongType(), True),
    StructField("rows_quarantined", LongType(), True),
    StructField("error_message", StringType(), True),
])

dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental"])
dbutils.widgets.text("processing_date", "")
dbutils.widgets.text("source_folder", "")

silver_load_type = dbutils.widgets.get("load_type")
silver_processing_date = dbutils.widgets.get("processing_date").strip()
silver_source_folder = dbutils.widgets.get("source_folder").strip()

if silver_load_type not in ("full", "incremental"):
    raise ValueError("load_type must be 'full' or 'incremental'.")

if silver_processing_date:
    try:
        date.fromisoformat(silver_processing_date)
    except ValueError:
        raise ValueError("processing_date must use YYYY-MM-DD format.")
elif silver_load_type == "incremental":
    silver_processing_date = datetime.now(timezone.utc).date().isoformat()
    print("No processing_date given; using today (UTC):", silver_processing_date)

spark.conf.set("spark.sql.session.timeZone", "UTC")

silver_run_id = str(uuid.uuid4())
silver_start_time = datetime.now(timezone.utc).replace(tzinfo=None)

bronze_table_name = "workspace.default.bronze_coin_market_raw"
silver_table_name = "workspace.default.silver_market_observation"
quarantine_table_name = "workspace.default.silver_quarantine"
execution_log_table_name = "workspace.default.pipeline_execution_logs"


def enforce_schema(df, schema):
    """Select and cast columns so the DataFrame matches the explicit schema exactly."""
    missing = [f.name for f in schema.fields if f.name not in df.columns]
    if missing:
        raise ValueError(f"Columns missing for schema enforcement: {missing}")
    return df.select([F.col(f.name).cast(f.dataType).alias(f.name) for f in schema.fields])


def keep_latest(df, order_col):
    """One row per business key; the row with the newest order_col wins."""
    w = Window.partitionBy("coin_id", "observation_timestamp").orderBy(
        F.col(order_col).desc_nulls_last()
    )
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")


print("Silver pipeline initialized. Run ID:", silver_run_id)
print("Load type:", silver_load_type)
print("Processing date:", silver_processing_date or "All dates")
print("Source folder:", silver_source_folder or "All Bronze sources")

In [ ]:
# Step 113: Show the explicit Silver schema (imported from schemas.py)
for field in silver_schema.fields:
    print(f"{field.name}: {field.dataType.simpleString()} (nullable={field.nullable})")
print("Silver business key: coin_id + observation_timestamp")

In [ ]:
# Step 114: Read the Bronze records selected by the parameters
bronze_df = spark.table(bronze_table_name)

if silver_processing_date:
    d = date.fromisoformat(silver_processing_date)
    obs_date = F.to_date(F.col("observation_timestamp"))
    if silver_load_type == "incremental":
        # The downloader fetches [run_date - 1, run_date], so include the overlap day.
        bronze_df = bronze_df.filter(obs_date.between(F.lit(d - timedelta(days=1)), F.lit(d)))
    else:
        bronze_df = bronze_df.filter(obs_date == F.lit(d))

if silver_source_folder:
    bronze_df = bronze_df.filter(F.col("source_file").contains(silver_source_folder))

silver_rows_read = bronze_df.count()
print("Bronze records selected:", silver_rows_read)
if silver_rows_read == 0:
    print("WARNING: no Bronze rows matched these parameters. Did Bronze load this date/folder?")

In [ ]:
# Step 115: Safe type conversion + data-quality validation
# try_cast returns NULL for bad values instead of failing (serverless runs with ANSI mode on),
# so type drift in Bronze ends up in quarantine and does not crash the batch.
typed_df = (
    bronze_df
    .withColumn("coin_id", F.col("coin_id").cast("string"))
    .withColumn("timestamp_ms", F.expr("try_cast(timestamp_ms AS BIGINT)"))
    .withColumn("observation_timestamp", F.expr("try_cast(observation_timestamp AS TIMESTAMP)"))
    .withColumn("price_usd", F.expr("try_cast(price_usd AS DOUBLE)"))
    .withColumn("market_cap_usd", F.expr("try_cast(market_cap_usd AS DOUBLE)"))
    .withColumn("total_volume_usd", F.expr("try_cast(total_volume_usd AS DOUBLE)"))
)

def bad(col):
    return F.col(col).isNull() | F.isnan(col) | (F.col(col) < 0)

silver_invalid_condition = (
    F.col("coin_id").isNull()
    | (F.trim(F.col("coin_id")) == "")
    | F.col("observation_timestamp").isNull()
    | bad("price_usd")
    | bad("market_cap_usd")
    | bad("total_volume_usd")
)

silver_valid_df = typed_df.filter(~silver_invalid_condition)
silver_invalid_df = typed_df.filter(silver_invalid_condition)

silver_valid_count = silver_valid_df.count()
silver_invalid_count = silver_invalid_df.count()

print("Total Bronze records:", silver_rows_read)
print("Valid:", silver_valid_count, "| Invalid:", silver_invalid_count)

if silver_valid_count + silver_invalid_count != silver_rows_read:
    raise ValueError("Data-quality validation failed: record counts do not match.")
print("Silver data-quality validation successful!")

In [ ]:
# Step 116: Rejection reasons
silver_invalid_details_df = silver_invalid_df.withColumn(
    "rejection_reason",
    F.concat_ws(
        "; ",
        F.when(F.col("coin_id").isNull() | (F.trim(F.col("coin_id")) == ""), "Invalid coin ID"),
        F.when(F.col("observation_timestamp").isNull(), "Invalid observation timestamp"),
        F.when(bad("price_usd"), "Invalid price"),
        F.when(bad("market_cap_usd"), "Invalid market cap"),
        F.when(bad("total_volume_usd"), "Invalid total volume"),
    ),
)
display(silver_invalid_details_df.select("coin_id", "observation_timestamp", "market_cap_usd", "rejection_reason").limit(50))

In [ ]:
# Step 117: MERGE invalid rows into the quarantine table (idempotent)
quarantine_candidates_df = silver_invalid_details_df.select(
    "coin_id", "source_file", "timestamp_ms", "observation_timestamp",
    "price_usd", "market_cap_usd", "total_volume_usd", "rejection_reason",
    F.col("load_timestamp").alias("bronze_load_timestamp"),
    F.lit(datetime.now(timezone.utc).replace(tzinfo=None)).alias("load_timestamp"),
    F.lit(silver_run_id).alias("run_id"),
)
silver_quarantine_source_df = enforce_schema(
    keep_latest(quarantine_candidates_df, "bronze_load_timestamp"), quarantine_schema
)

(
    DeltaTable.forName(spark, quarantine_table_name).alias("target")
    .merge(
        silver_quarantine_source_df.alias("source"),
        "target.coin_id <=> source.coin_id AND target.observation_timestamp <=> source.observation_timestamp",
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
print("Quarantine MERGE done. Total quarantine rows:", spark.table(quarantine_table_name).count())

In [ ]:
# Step 118: Prepare valid rows (de-duplicate on the business key; latest Bronze row wins)
silver_load_ts = datetime.now(timezone.utc).replace(tzinfo=None)

silver_candidates_df = silver_valid_df.select(
    "coin_id", "observation_timestamp", "price_usd", "market_cap_usd", "total_volume_usd",
    "source_file",
    F.col("load_timestamp").alias("bronze_load_timestamp"),
    F.lit(silver_load_ts).alias("load_timestamp"),
    F.lit(silver_run_id).alias("run_id"),
)
silver_source_df = enforce_schema(keep_latest(silver_candidates_df, "bronze_load_timestamp"), silver_schema)

silver_prepared_count = silver_source_df.count()
silver_duplicates_dropped = silver_valid_count - silver_prepared_count
print("Valid rows prepared for MERGE:", silver_prepared_count)
print("Duplicate business keys collapsed:", silver_duplicates_dropped)

In [ ]:
# Step 119: Idempotent MERGE into Silver
silver_before_count = spark.table(silver_table_name).count()
silver_delta = DeltaTable.forName(spark, silver_table_name)
version_before = silver_delta.history(1).select("version").first()[0]

measures = ["price_usd", "market_cap_usd", "total_volume_usd"]
changed_condition = " OR ".join(f"NOT (target.{c} <=> source.{c})" for c in measures)

(
    silver_delta.alias("target")
    .merge(
        silver_source_df.alias("source"),
        "target.coin_id = source.coin_id AND target.observation_timestamp = source.observation_timestamp",
    )
    .whenMatchedUpdate(
        condition=changed_condition,          # re-running identical data updates nothing
        set={
            "price_usd": "source.price_usd",
            "market_cap_usd": "source.market_cap_usd",
            "total_volume_usd": "source.total_volume_usd",
            "source_file": "source.source_file",
            "bronze_load_timestamp": "source.bronze_load_timestamp",
            "load_timestamp": "source.load_timestamp",
            "run_id": "source.run_id",
        },
    )
    .whenNotMatchedInsertAll()
    .execute()
)

latest = silver_delta.history(1).first()
if latest["version"] > version_before and latest["operation"] == "MERGE":
    metrics = latest["operationMetrics"]
    silver_rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
    silver_rows_updated = int(metrics.get("numTargetRowsUpdated", 0))
else:
    silver_rows_inserted, silver_rows_updated = 0, 0   # MERGE changed nothing

silver_after_count = spark.table(silver_table_name).count()
print("Silver before:", silver_before_count, "| after:", silver_after_count)
print("Rows inserted:", silver_rows_inserted, "| Rows updated:", silver_rows_updated)

In [ ]:
# Step 120: Write the Silver execution log (one row per run)
silver_end_time = datetime.now(timezone.utc).replace(tzinfo=None)

silver_log_df = spark.createDataFrame(
    [(
        silver_run_id, "silver", silver_table_name, silver_load_type,
        silver_processing_date or None, silver_source_folder or None,
        silver_start_time, silver_end_time, "SUCCESS",
        silver_rows_read, silver_rows_inserted, silver_rows_updated,
        silver_invalid_count, None,
    )],
    schema=execution_log_schema,
)

(
    DeltaTable.forName(spark, execution_log_table_name).alias("target")
    .merge(silver_log_df.alias("source"), "target.run_id = source.run_id")
    .whenNotMatchedInsertAll()
    .execute()
)

summary = {
    "run_id": silver_run_id, "status": "SUCCESS", "rows_read": silver_rows_read,
    "rows_inserted": silver_rows_inserted, "rows_updated": silver_rows_updated,
    "rows_quarantined": silver_invalid_count,
}
print(summary)
display(spark.table(execution_log_table_name).filter(F.col("run_id") == silver_run_id))

dbutils.notebook.exit(json.dumps(summary))